# Day 1 — vLLM：跑通 + 核心对象与链路

## 今日完成标准
- [ ] 本机 vLLM 0.8.5 能成功生成文本（离线调用 LLM.generate）
- [ ] 能解释 SamplingParams 的 5 个常用字段：temperature/top_p/top_k/max_tokens/stop
- [ ] 能用一次实验观察 prefill 与 decode（用输出 token 数与时间近似对比）
- [ ] 能定位 vllm 安装目录并找到 engine/scheduler 相关源码文件位置


In [1]:
import torch, sys, os, platform

print("python:", sys.version)
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda version (torch):", torch.version.cuda)
    print("gpu:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free/total: {free/1024**3:.2f} / {total/1024**3:.2f} GB")

import vllm
print("vllm:", vllm.__version__)


python: 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
platform: Linux-5.15.0-168-generic-x86_64-with-glibc2.35
torch: 2.6.0+cu118
cuda available: True
cuda version (torch): 11.8
gpu: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM free/total: 5.06 / 5.79 GB


/home/wang/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 02-07 22:48:00 [__init__.py:239] Automatically detected platform cuda.
vllm: 0.8.5


In [2]:
import vllm, inspect

print("vLLM version:", getattr(vllm, "__version__", "unknown"))

root = os.path.dirname(inspect.getfile(vllm))
print("vLLM package root:", root)

# 你今天要“记住”的三个目录：engine / core / worker（不同版本命名略有差异）
candidates = [
    ("engine", "vllm/engine"),
    ("core", "vllm/core"),
    ("worker", "vllm/worker"),
    ("entrypoints", "vllm/entrypoints"),
]
for name, rel in candidates:
    path = os.path.join(root, rel.split("/",1)[-1])
    print(f"{name:10s} ->", path, "exists" if os.path.exists(path) else "MISSING")


vLLM version: 0.8.5
vLLM package root: /home/wang/anaconda3/envs/llm/lib/python3.11/site-packages/vllm
engine     -> /home/wang/anaconda3/envs/llm/lib/python3.11/site-packages/vllm/engine exists
core       -> /home/wang/anaconda3/envs/llm/lib/python3.11/site-packages/vllm/core exists
worker     -> /home/wang/anaconda3/envs/llm/lib/python3.11/site-packages/vllm/worker exists
entrypoints -> /home/wang/anaconda3/envs/llm/lib/python3.11/site-packages/vllm/entrypoints exists


## vLLM 最小心智模型（Day1 版）

用户请求 -> (Prompt)  
-> Engine 接收 request  
-> Scheduler 决定谁 prefill / 谁 decode（并发、抢占、batching）  
-> Worker/GPU 执行 attention（PagedAttention + KV Cache）  
-> 输出 token 流式返回 / 或一次性返回

今天只做：从“用户调用 API”走到“Engine 调用成功”，并在源码里定位入口文件。


In [3]:
from vllm import LLM, SamplingParams

# ✅ 把它换成你本机能用的模型（例如：Qwen2.5-1.5B-Instruct / Llama-3.2-1B-Instruct 等）
MODEL = os.environ.get("VLLM_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

sampling = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=128,
)

prompts = [
    "用三句话解释什么是 KV cache，并说明它为什么能加速解码阶段。",
    "给我一个一句话定义：大模型当中prefill 和 decode 的区别是什么？",
]

# 6GB 显存建议：限制并发与长度
llm = LLM(
    model=MODEL,
    tensor_parallel_size=1,
    # enforce_eager=True,
    # ① 降上下文长度 => KV cache 直接变小
    max_model_len=512,

    # ② 降“并发序列数” => 直接避免 warmup 256 dummy req 爆显存
    max_num_seqs=1,

    # ③ 降可用显存比例 => 给 torch.compile / graph capture 留空间
    gpu_memory_utilization=0.8,

)


outs = llm.generate(prompts, sampling)
for i, o in enumerate(outs):
    print("="*80)
    print("PROMPT:", prompts[i])
    print("OUTPUT:", o.outputs[0].text)


INFO 02-07 22:48:17 [config.py:717] This model supports multiple tasks: {'classify', 'score', 'generate', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 02-07 22:48:17 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 02-07 22:48:20 [utils.py:2382] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 02-07 22:48:22 [__init__.py:239] Automatically detected platform cuda.
INFO 02-07 22:48:24 [core.py:58] Initializing a V1 LLM engine (v0.8.5) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=512, download_dir=None,

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  6.51it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  6.50it/s]



INFO 02-07 22:48:26 [gpu_model_runner.py:1347] Model loading took 0.9277 GiB and 2.016222 seconds
INFO 02-07 22:48:30 [backends.py:420] Using cache directory: /home/wang/.cache/vllm/torch_compile_cache/307a0892db/rank_0_0 for vLLM's torch.compile
INFO 02-07 22:48:30 [backends.py:430] Dynamo bytecode transform time: 4.04 s
INFO 02-07 22:48:33 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 2.484 s
INFO 02-07 22:48:34 [monitor.py:33] torch.compile takes 4.04 s in total
INFO 02-07 22:48:34 [kv_cache_utils.py:634] GPU KV cache size: 222,576 tokens
INFO 02-07 22:48:34 [kv_cache_utils.py:637] Maximum concurrency for 512 tokens per request: 434.72x
INFO 02-07 22:48:46 [gpu_model_runner.py:1686] Graph capturing finished in 12 secs, took 1.20 GiB
INFO 02-07 22:48:46 [core.py:159] init engine (profile, create kv cache, warmup model) took 19.64 seconds
INFO 02-07 22:48:46 [core_client.py:439] Core engine process 0 ready.


Processed prompts: 100%|██████████| 2/2 [00:01<00:00,  1.33it/s, est. speed input: 22.03 toks/s, output: 170.88 toks/s]

PROMPT: 用三句话解释什么是 KV cache，并说明它为什么能加速解码阶段。
OUTPUT:  在计算机系统中，缓存是一种存储数据的内存区域。在解码阶段，我们通常需要查找解码后的数据，但是内存通常是有限的，无法存储所有的数据。因此，我们使用缓存来存储已经解码的数据，以便在需要时快速访问。KV cache 是一种使用哈希表来存储键值对的缓存技术，其中键是数据的唯一标识，值是数据本身。KV cache 的主要优点是能够加速解码阶段，因为我们可以直接从缓存中获取解码后的数据，而不是从内存中读取。这使得我们在解码阶段
PROMPT: 给我一个一句话定义：大模型当中prefill 和 decode 的区别是什么？
OUTPUT: prefill 和 decode 是在哪些场景下常用于实现预填充和解码的？

在自然语言处理中，prefill 和 decode 是两种常见的操作，它们在实现中具有不同的特点和应用场景。以下是它们的区别以及在自然语言处理中常用于实现的场景：

1. prefill：
prefill 是一种将输入文本中的空白或空位填充到适当位置的操作。prefill 通常在文本预处理阶段使用，如在文本分类、词性标注、分词等任务中。prefill 使得模型能够更好地理解和处理预处理后的文本，提高模型的性能。

2


In [4]:
from vllm import SamplingParams

def show_sampling(params: SamplingParams):
    keys = ["temperature","top_p","top_k","max_tokens","stop",
            "presence_penalty","frequency_penalty"]
    return {k: getattr(params, k, None) for k in keys}

tests = [
    SamplingParams(temperature=0.0, max_tokens=64),
    SamplingParams(temperature=0.8, top_p=0.9, max_tokens=64),
    SamplingParams(temperature=0.8, top_k=50, max_tokens=64),
    SamplingParams(temperature=0.7, top_p=0.9, stop=["\n"], max_tokens=128),
]

for t in tests:
    print(show_sampling(t))

{'temperature': 0.0, 'top_p': 1.0, 'top_k': -1, 'max_tokens': 64, 'stop': [], 'presence_penalty': 0.0, 'frequency_penalty': 0.0}
{'temperature': 0.8, 'top_p': 0.9, 'top_k': -1, 'max_tokens': 64, 'stop': [], 'presence_penalty': 0.0, 'frequency_penalty': 0.0}
{'temperature': 0.8, 'top_p': 1.0, 'top_k': 50, 'max_tokens': 64, 'stop': [], 'presence_penalty': 0.0, 'frequency_penalty': 0.0}
{'temperature': 0.7, 'top_p': 0.9, 'top_k': -1, 'max_tokens': 128, 'stop': ['\n'], 'presence_penalty': 0.0, 'frequency_penalty': 0.0}


In [12]:
import time
from vllm import SamplingParams

short_prompt = "解释一下 vLLM 的调度器做什么。"
long_prompt = (
    "请逐条解释 vLLM 推理过程中的关键阶段（从请求进入到 token 输出）：\n"
    + "\n".join([f"{i}. 这是第{i}条补充背景信息，用于拉长输入但保持语义多样。" for i in range(1, 20)])
)
def run(prompt, max_tokens):
    sp = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=max_tokens)
    t0 = time.time()
    out = llm.generate([prompt], sp)[0].outputs[0].text
    t1 = time.time()
    return t1 - t0, out

cases = [
    ("short_prompt + short_decode", short_prompt, 4),
    ("short_prompt + long_decode",  short_prompt, 256),
    ("long_prompt + short_decode",  long_prompt, 4),
]

for name, p, mt in cases:
    dt, preview = run(p, mt)
    print(f"{name:28s} | max_tokens={mt:3d} | time={dt:.3f}s | preview={preview!r}")


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 30.62it/s, est. speed input: 307.35 toks/s, output: 122.86 toks/s]


short_prompt + short_decode  | max_tokens=  4 | time=0.035s | preview='VLLM 是'


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it, est. speed input: 6.46 toks/s, output: 165.36 toks/s]


short_prompt + long_decode   | max_tokens=256 | time=1.550s | preview=' VLLM（Virtual Language Service）调度器是一种用于调度虚拟语言服务的组件，它负责处理和管理虚拟语言服务的请求。VLLM调度器的主要功能包括：\n\n1. 任务调度：根据调度策略，为虚拟语言服务分配任务，调度任务以最优化的方式处理请求。\n\n2. 任务执行：在执行任务时，VLLM调度器负责将请求分配给合适的虚拟语言服务，确保任务的高效处理。\n\n3. 任务监控：VLLM调度器提供任务状态的监控功能，包括任务状态的更新、任务执行的记录和状态报告，以便管理员可以实时了解任务的运行情况。\n\n4. 任务优化：VLLM调度器可以对任务进行优化，例如根据任务的复杂度、资源消耗、任务执行的延迟等参数，自动选择最优的虚拟语言服务进行处理，以提高处理效率和资源利用效率。\n\n5. 任务通知：VLLM调度器可以将任务的状态和执行结果通知给相关用户或系统，以便他们可以及时了解任务的进展和结果。\n\n6. 任务资源管理：VLLM调度器可以自动管理和分配虚拟语言服务的资源，例如计算资源、内存资源'


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 25.75it/s, est. speed input: 11878.88 toks/s, output: 103.24 toks/s]

long_prompt + short_decode   | max_tokens=  4 | time=0.041s | preview=' 这是第'


In [13]:
tok = llm.get_tokenizer()
def n_tok(s): 
    return len(tok.encode(s))

print("short_prompt tokens:", n_tok(short_prompt))
print("long_prompt  tokens:", n_tok(long_prompt))


short_prompt tokens: 10
long_prompt  tokens: 460


In [12]:
import os, glob, re

def find_files(root, patterns):
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(os.path.join(root, "**", pat), recursive=True))
    return sorted(set(hits))

patterns = [
    "*engine*.py",
    "*scheduler*.py",
    "*llm_engine*.py",
    "*sequence*.py",
]
hits = find_files(root, patterns)

print("Found", len(hits), "candidate files.")
for p in hits[:40]:
    print(p.replace(root, "vllm"))


Found 10 candidate files.
vllm/compilation/sequence_parallelism.py
vllm/core/scheduler.py
vllm/engine/async_llm_engine.py
vllm/engine/llm_engine.py
vllm/engine/multiprocessing/engine.py
vllm/entrypoints/openai/serving_engine.py
vllm/sequence.py
vllm/v1/core/sched/scheduler.py
vllm/v1/engine/llm_engine.py
vllm/worker/cache_engine.py


## Day1 口述模板（背下来）

**1) vLLM 做什么？**
vLLM 是面向大模型推理的高吞吐引擎，核心通过高效的 request batching + 调度 + PagedAttention(KV cache 分页管理) 来提升并发和显存利用率。

**2) prefill vs decode？**
prefill 是把 prompt 一次性送入模型并填充 KV cache（成本与 prompt 长度相关）；decode 是在已有 KV cache 上逐 token 生成（成本与生成 token 数相关）。

**3) KV cache 为什么加速？**
Transformer 自回归生成时，历史 K/V 不需要重复计算；缓存下来后，每步 decode 只计算新 token 的 Q/K/V 并与缓存做 attention，避免 O(T^2) 的重复前向。
